# 05 — Reliability-qualified NTL: 2012–2024 sensitivity

This notebook builds the exact revision sequence: (1) full-period regional direct DNB-BRDF, Gap-Filled, and Román-inspired four-day comparison with $r$, paired z-score RMSE, and retained $n$; (2) SC and GHSL qualification; (3) 1–30-calendar-day sensitivity curves and spatial-scale × window heatmaps at declared SC thresholds; (4) composite-weight sensitivity and held-out island-grid evaluation. Haiyan and Rai appear only as zoomed examples. **This notebook is unexecuted; run it on your RQ1 data and interpret the figures.**

## Data contract

The study clock is **19 January 2012–31 December 2024** throughout. The opening direct and Gap-Filled signals come from the **same Region VIII CSV you identified**, `Region VIII_NTL_VNP46A2.csv`, using `DNB_BRDF_Corrected_NTL` and `Gap_Filled_DNB_BRDF_Corrected_NTL`. The notebook searches the RQ1 project for this file; if it lives elsewhere, set `RQNTL_REGION_VIII_CSV` to its full path. Thus the raw–filled comparison has the same dates and exported regional footprint. The RQ1 direct regional `EasternVisayas_DNBBRDF_daily_stats.csv` supplies valid-pixel support and the 1–30-day sensitivity; eight GHSL masks and 1 AM NGCP load follow RQ1 Notebooks 03–04. No RQ2 data are loaded.

All primary NTL metrics use the RQ1-style **0th–95th percentile range**: the upper 5% is capped at the 95th percentile within each signal; uncapped columns are retained for audit. This cap is stated in every relevant figure. The four-day signal is a **non-overlapping mean of the companion CSV's direct regional daily series**, anchored to Haiyan as in the sample RQ2 notebook. RQ2 uses a pixel-first four-day mean and a recovery baseline; those cannot be exactly reconstructed from the RQ1 daily summary CSV. This is an explicitly labelled Román-inspired temporal analogue, not a pixel-first replication. Raw and Gap-Filled $n$ count paired **days**; four-day $n$ counts complete calendar **blocks**. All RMSE values use paired z-scores, so $\mathrm{RMSE}^2=2(1-r)$.


In [ ]:
from pathlib import Path
import os, re, math, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.max_columns = 100
ROOT = Path(os.environ.get('RQNTL_PROJECT_ROOT', Path.cwd())).expanduser().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'datasets'
def find_region_viii_csv():
    override = os.environ.get('RQNTL_REGION_VIII_CSV')
    if override:
        p = Path(override).expanduser()
        if not p.is_file():
            raise FileNotFoundError(f'RQNTL_REGION_VIII_CSV does not exist: {p}')
        return p
    name = 'Region VIII_NTL_VNP46A2.csv'
    candidates = [DATA / 'vnp46a2/region8_daily_stats' / name,
                  DATA / 'vnp46a2' / name, DATA / name, ROOT / name]
    for p in candidates:
        if p.is_file(): return p
    for base in (DATA, ROOT, ROOT.parent):
        if not base.is_dir(): continue
        try:
            found = next(base.rglob(name), None)
        except OSError:
            found = None
        if found is not None: return found
    raise FileNotFoundError(f'{name} was not found under the RQ1 project. Set RQNTL_REGION_VIII_CSV to its full path.')

REGION_VIII_CSV = find_region_viii_csv()
REGIONAL_CSV = DATA / 'vnp46a2/region8_daily_stats/EasternVisayas_DNBBRDF_daily_stats.csv'
GHSL_DIR = DATA / 'vnp46a2/ghsl_masked_ntl_samar_leyte'
CROSS_DIR = DATA / 'vnp46a2/visayas_ghsl_masked_daily_stats'
NGCP_CSV = DATA / 'ngcp/NGCP_Hourly_Demand.csv'
NGCP_XLSX = DATA / 'ngcp/NGCP_hourly_load.xlsx'
OUT = Path(os.environ.get('RQNTL_OUTPUT_ROOT', ROOT / 'outputs/sensitivity/revision05')).expanduser().resolve()
OUT.mkdir(parents=True, exist_ok=True)

START, END = pd.Timestamp('2012-01-19'), pd.Timestamp('2024-12-31')
CALENDAR = pd.date_range(START, END, freq='D', name='date')
HAIYAN = pd.Timestamp('2013-11-08')
WINDOWS = list(range(1, 31))
TAUS = [0, 5, 10, 20, 30, 50, 70, 90]
SC_SWEEP_TAUS = list(range(101))  # dense 0–100% SC plot, following RQ1 Notebook 03
TRANSFER_TAUS = [0, 5, 10, 20, 30, 50, 70]
NTL_UPPER_PERCENTILE = 0.95  # same upper-tail cap for direct and Gap-Filled NTL
MIN_WINDOW_FRACTION = 0.50  # only for the separately labelled window-supported scenario
PLATEAU_DELTA_R = 0.02       # sensitivity is repeated below for 0.01 and 0.05
MIN_DIRECT_COVERAGE = 0.20  # eligibility rule for transfer training islands
MIN_PAIRS = 5
GROUPS = {
    '30': ('G1', 'Urban Core', '#D65A4A'),
    '23-30': ('G2', 'Dense Urban', '#A87000'),
    '22-30': ('G3', 'Core & Inner', '#E6A23C'),
    '21-30': ('G4', 'Urban Fringe', '#D9B900'),
    '13-30': ('G5', 'Built-Up / Rural Mix', '#375623'),
    '12-30': ('G6', 'Low–Mid Rural', '#7BAF54'),
    '11-30': ('G7', 'Rural Settlements', '#A8CE74'),
    '10-30': ('G8', 'All Land GHSL classes', '#4E79A7'),
}
FULL_LABEL = 'Study area (unmasked)'
MASK_ORDER = [v[0] for v in GROUPS.values()] + [FULL_LABEL]  # G8 beside unmasked area
COLOR = {FULL_LABEL: '#183A63', **{v[0]: v[2] for v in GROUPS.values()}}
print('RQ1 root:', ROOT)
print('Same-footprint Region VIII raw + Gap-Filled CSV:', REGION_VIII_CSV)
print('Long-record full-area input:', REGIONAL_CSV)
print('Output folder:', OUT)

def save_fig(fig, name):
    fig.update_layout(template='plotly_white',paper_bgcolor='rgba(0,0,0,0)',
                      font=dict(color='#233B60',size=13))
    fig.show()
    fig.write_html(OUT / (name + '.html'), include_plotlyjs='cdn')
    try:
        fig.write_image(OUT / (name + '.png'), scale=2)
    except Exception as exc:
        print(f'{name}: HTML saved; PNG skipped ({type(exc).__name__}). Install kaleido for PNG.')

def date_from_frame(frame):
    d = frame.copy()
    if 'date' not in d:
        if 'system:index' not in d:
            raise ValueError('Expected date or system:index')
        d['date'] = d['system:index'].astype(str).str.extract(r'(\d{4}_\d{2}_\d{2})')[0].str.replace('_','-',regex=False)
    d['date'] = pd.to_datetime(d['date'], errors='coerce').dt.normalize()
    if d['date'].isna().any() or d['date'].duplicated().any():
        raise ValueError('Missing or duplicate dates in an NTL input')
    return d.set_index('date').sort_index()

def ngcp_hour1(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    f = pd.read_csv(path, skiprows=2)
    if f.shape[1] < 4:
        raise ValueError('NGCP CSV layout differs from RQ1 Notebook 03')
    f.columns = ['Date','Coincident Peak','Hour No'] + [str(i) for i in range(1, f.shape[1]-2)]
    f['Date'] = pd.to_datetime(f['Date'], dayfirst=True, errors='coerce').dt.normalize()
    f['1'] = pd.to_numeric(f['1'], errors='coerce')
    f = f.dropna(subset=['Date']).set_index('Date').sort_index()
    if f.index.duplicated().any():
        raise ValueError('Duplicate NGCP dates')
    return f['1'].reindex(CALENDAR).rename('ngcp_1am_mw')

LOAD = ngcp_hour1(NGCP_CSV)

def paired_metrics(signal, target, eligible_index=None):
    x, y = pd.Series(signal, dtype=float), pd.Series(target, dtype=float)
    if eligible_index is None:
        eligible_index = y.index[y.notna()]
    eligible_index = pd.Index(eligible_index)
    p = pd.concat([x.rename('ntl'), y.rename('load')], axis=1).reindex(eligible_index).dropna()
    n, possible = len(p), len(eligible_index)
    result = {'n_pairs': n, 'n_possible': possible,
              'n_pct': 100*n/possible if possible else np.nan,
              'r': np.nan, 'rmse_z': np.nan}
    if n < MIN_PAIRS or p.ntl.std(ddof=0) == 0 or p.load.std(ddof=0) == 0:
        return result
    a = (p.ntl-p.ntl.mean()) / p.ntl.std(ddof=0)
    b = (p.load-p.load.mean()) / p.load.std(ddof=0)
    result['r'] = float(a.corr(b))
    result['rmse_z'] = float(np.sqrt(np.mean((a-b)**2)))
    return result

## 1. The three full-period signals before reliability qualification

**Purpose:** compare direct DNB-BRDF, its Gap-Filled companion, and a four-day direct composite for all of 2012–2024. The first two are columns from the **same Region VIII export**. The charts let us test whether direct radiance is erratic, whether apparent Gap-Filled continuity occurs when direct observations are sparse, and whether four-day aggregation improves load alignment. These are questions for the results, not conclusions assumed in advance.


In [ ]:
companion=date_from_frame(pd.read_csv(REGION_VIII_CSV))
required={'DNB_BRDF_Corrected_NTL','Gap_Filled_DNB_BRDF_Corrected_NTL'}
missing=required-set(companion.columns)
if missing:
    raise ValueError(f'{REGION_VIII_CSV.name}: missing {sorted(missing)}')
DIRECT_UNCAPPED=pd.to_numeric(companion['DNB_BRDF_Corrected_NTL'],errors='coerce').reindex(CALENDAR)
GAP_UNCAPPED=pd.to_numeric(companion['Gap_Filled_DNB_BRDF_Corrected_NTL'],errors='coerce').reindex(CALENDAR)
if DIRECT_UNCAPPED.notna().sum()<365 or GAP_UNCAPPED.notna().sum()<365:
    raise ValueError('The Region VIII CSV must contain the long RQ1 daily record, not an event-only subset.')
def cap_0_95(s):
    valid=s.dropna()
    return s.clip(lower=valid.quantile(0),upper=valid.quantile(NTL_UPPER_PERCENTILE))
DIRECT_RAW=cap_0_95(DIRECT_UNCAPPED)
GAP_RAW=cap_0_95(GAP_UNCAPPED)
print('NTL P95 caps: direct',round(DIRECT_UNCAPPED.quantile(.95),4),
      'Gap-Filled',round(GAP_UNCAPPED.quantile(.95),4))
GAP_AVAILABLE=True
# SC is supplied by RQ1 regional direct statistics, not inferred from filled values.
direct_frame=date_from_frame(pd.read_csv(REGIONAL_CSV))
if not {'NTL_mean','Valid_px'} <= set(direct_frame.columns):
    raise ValueError('RQ1 regional CSV must have NTL_mean and Valid_px')
DIRECT_PX=pd.to_numeric(direct_frame.Valid_px,errors='coerce').reindex(CALENDAR)
DIRECT_SC=100*DIRECT_PX/DIRECT_PX.max()

# Event-anchored, non-overlapping, complete four-calendar-day units.
block_id=np.floor_divide((CALENDAR-HAIYAN).days,4)
block_start=pd.DatetimeIndex(HAIYAN+pd.to_timedelta(np.unique(block_id)*4,unit='D'))
block_table=pd.DataFrame({'direct':DIRECT_RAW.to_numpy(),'gap':GAP_RAW.to_numpy(),
                          'load':LOAD.to_numpy(),'block':block_id},index=CALENDAR)
agg=block_table.groupby('block').agg(direct_mean=('direct','mean'),gap_mean=('gap','mean'),
                                      load_mean=('load','mean'),direct_days=('direct','count'),
                                      gap_days=('gap','count'),load_days=('load','count'),calendar_days=('load','size'))
agg.index=block_start
agg=agg[(agg.calendar_days==4)&(agg.load_days==4)]
agg['four_day_direct']=agg.direct_mean.where(agg.direct_days>0)
agg.to_csv(OUT/'opening_fullperiod_four_day_block_support.csv')
ELIGIBLE_DAYS=CALENDAR[LOAD.notna()]
ELIGIBLE_BLOCKS=agg.index
OPENING=pd.DataFrame([
 {'signal':'Direct DNB-BRDF (P95)','unit':'days',**paired_metrics(DIRECT_RAW,LOAD,ELIGIBLE_DAYS)},
 {'signal':'Gap-Filled (P95)','unit':'days',**paired_metrics(GAP_RAW,LOAD,ELIGIBLE_DAYS)},
 {'signal':'4-day direct mean (P95)','unit':'4-day blocks',
  **paired_metrics(agg.four_day_direct,agg.load_mean,ELIGIBLE_BLOCKS)}])
OPENING.to_csv(OUT/'opening_2012_2024_three_signal_metrics.csv',index=False)
print('Study calendar:',CALENDAR.min().date(),'to',CALENDAR.max().date())
print('Direct + Gap-Filled source:',REGION_VIII_CSV)
display(OPENING.round(3))

### Figure 1A — $r$, RMSE and N%, 2012–2024; NTL capped at P95

The first two denominators are load-available days; the third is complete four-day blocks with all four NGCP days. The metrics use their native clocks, so compare the broad pattern and availability rather than treating a daily $n$ as equivalent to a block $n$. For a common-clock diagnostic, the CSV below also aggregates raw and Gap-Filled daily regional values to the same four-day blocks.

In [ ]:
common=pd.DataFrame([
 {'signal':'Direct on common 4-day blocks',**paired_metrics(agg.direct_mean,agg.load_mean,ELIGIBLE_BLOCKS)},
 {'signal':'Gap-Filled on common 4-day blocks',**paired_metrics(agg.gap_mean,agg.load_mean,ELIGIBLE_BLOCKS)},
 {'signal':'Four-day direct regional composite',**paired_metrics(agg.four_day_direct,agg.load_mean,ELIGIBLE_BLOCKS)}])
common.to_csv(OUT/'supplement_fullperiod_common_four_day_clock.csv',index=False)
if GAP_AVAILABLE:
    colors=['#4E79A7','#D65A4A','#375623']
    fig=make_subplots(rows=1,cols=3,subplot_titles=['Pearson r','Paired z-score RMSE','Retained analysis units (%)'])
    for col,field in enumerate(['r','rmse_z','n_pct'],1):
        vals=OPENING[field].to_numpy(float)
        labels=[f'{int(x.n_pairs)}/{int(x.n_possible)} {x.unit}' for x in OPENING.itertuples()]
        fig.add_trace(go.Bar(x=OPENING.signal,y=vals,marker_color=colors,showlegend=False,
                             text=labels if field=='n_pct' else [f'{v:.2f}' if np.isfinite(v) else 'undefined' for v in vals],
                             textposition='outside'),row=1,col=col)
    fig.update_yaxes(title_text='r',row=1,col=1)
    fig.update_yaxes(title_text='RMSE (paired SD units)',row=1,col=2)
    fig.update_yaxes(title_text='n / eligible (%)',range=[0,115],row=1,col=3)
    fig.update_xaxes(tickangle=-18)
    fig.update_layout(template='plotly_white',width=1370,height=500,
                      title='Study area, 2012–2024: P95-capped NTL alignment and N%',
                      margin=dict(l=65,r=30,t=85,b=145),font=dict(color='#233B60'))
    save_fig(fig,'fig_01a_fullperiod_three_signal_metrics')

### Figure 1B — The full period first; storm windows are zooms

The full 2012–2024 plot keeps the three P95-capped NTL signals in context. The top strip is directly valid-pixel support from the raw band; high Gap-Filled availability is not new direct evidence. The four-day value is repeated only as a **step display** across its own block, never counted as four independent observations. Haiyan and Rai zooms use the same underlying series and are illustrative, not separate validation samples.

In [ ]:
# Keep the preceding data preparation and metrics-table code.
PLOT_RIGHT = COMMON_END

def p95_for_display(series):
    observed = series.loc[FULL_LEFT:PLOT_RIGHT].dropna()
    return series.clip(upper=observed.quantile(.95))

PLOT_SERIES = {
    'direct': p95_for_display(DIRECT),
    'filled': p95_for_display(FILLED),
    'area4': p95_for_display(DIRECT_4D),
    'rq4': p95_for_display(RQ_4D),
}
PLOT_LIMIT = {
    2: 1.07 * max(PLOT_SERIES['direct'].max(),
                  PLOT_SERIES['filled'].max()),
    4: 1.07 * max(PLOT_SERIES['area4'].max(),
                  PLOT_SERIES['rq4'].max()),
}
AREA_COLOR = '#2952CC'
RQ_COLOR = '#0B7A3E'

def panel_z(series, left, right):
    visible = series.loc[left:right]
    sd = visible.std(ddof=0)
    if not np.isfinite(sd) or sd == 0:
        return series * np.nan
    return (series - visible.mean()) / sd

def clean_sc_strip(fig, row, col, dates, label, values, colorbar=False):
    support = values.reindex(dates)
    hover = [
        'Missing' if pd.isna(value) else f'{value:.0f}%'
        for value in support
    ]

    fig.add_trace(
        go.Heatmap(
            x=dates,
            y=[label],
            z=[support.fillna(-1).to_numpy(float)],
            customdata=[hover],
            zmin=-1,
            zmax=100,
            colorscale=[
                [0, '#E6E6E6'],
                [.0098, '#E6E6E6'],
                [.0099, '#F7FCF5'],
                [.25, '#C7E9C0'],
                [.5, '#74C476'],
                [.75, '#238B45'],
                [1, '#00441B'],
            ],
            showscale=colorbar,
            colorbar=dict(
                title='SC %',
                x=.958,
                xanchor='left',
                y=.50,
                len=.15,
                thickness=12,
                xpad=0,
                tickvals=[0, 50, 100],
            ),
            hovertemplate=(
                '%{y}<br>%{x|%d %b %Y}<br>'
                'SC: %{customdata}<extra></extra>'
            ),
        ),
        row=row,
        col=col,
    )
    fig.update_xaxes(showticklabels=False, row=row, col=col)

def clean_signal(
    fig, row, col, clock, plotted, actual,
    label, color, legend_id, is_zoom, view, left, right
):
    y = plotted if view == 'radiance' else panel_z(
        plotted, left, right
    )

    if is_zoom:
        style = dict(
            mode='lines',
            line=dict(
                color=color,
                width=2.1 if label == 'RQ G3 4D' else 1.6,
                dash='dot' if label == 'Area 4D' else 'solid',
                shape='linear' if row == 2 else 'hv',
            ),
        )
    else:
        style = dict(
            mode='markers',
            marker=dict(
                color=color,
                size=2.5 if row == 2 else 3,
                opacity=.35,
            ),
        )

    fig.add_trace(
        go.Scatter(
            x=clock,
            y=y.reindex(clock),
            customdata=actual.reindex(clock),
            name=label,
            legend=legend_id,
            connectgaps=False,
            hovertemplate=(
                '%{x|%d %b %Y}<br>'
                'Actual radiance=%{customdata:.2f}'
                '<extra></extra>'
            ),
            **style,
        ),
        row=row,
        col=col,
        secondary_y=False,
    )

def clean_main_figure(view):
    fig = make_subplots(
        rows=4,
        cols=2,
        row_heights=[.055, .445, .055, .445],
        column_widths=[.57, .43],
        vertical_spacing=.035,
        horizontal_spacing=.07,
        specs=[
            [{'type': 'heatmap'}, {'type': 'heatmap'}],
            [{'secondary_y': True}, {'secondary_y': True}],
            [{'type': 'heatmap'}, {'type': 'heatmap'}],
            [{'secondary_y': True}, {'secondary_y': True}],
        ],
    )

    for col, left, right in [
        (1, FULL_LEFT, PLOT_RIGHT),
        (2, ZOOM_LEFT, ZOOM_RIGHT),
    ]:
        is_zoom = col == 2
        dates = CALENDAR[(CALENDAR >= left) & (CALENDAR <= right)]
        blocks = NGCP_4D.loc[left:right].index
        top_legend = 'legend' if col == 1 else 'legend2'
        bottom_legend = 'legend3' if col == 1 else 'legend4'

        clean_sc_strip(
            fig, 1, col, dates, 'Area', AREA_SC,
            colorbar=is_zoom,
        )
        clean_sc_strip(
            fig, 3, col, dates, 'G3', G3_SC,
        )

        for plotted, actual, label, color in [
            (PLOT_SERIES['direct'], DIRECT, 'Direct', COLOR['direct']),
            (PLOT_SERIES['filled'], FILLED, 'Gap-Filled', COLOR['filled']),
        ]:
            clean_signal(
                fig, 2, col, dates, plotted, actual,
                label, color, top_legend, is_zoom,
                view, left, right,
            )

        for plotted, actual, label, color in [
            (PLOT_SERIES['area4'], DIRECT_4D,
             'Area 4D', AREA_COLOR),
            (PLOT_SERIES['rq4'], RQ_4D,
             'RQ G3 4D', RQ_COLOR),
        ]:
            clean_signal(
                fig, 4, col, blocks, plotted, actual,
                label, color, bottom_legend, is_zoom,
                view, left, right,
            )

        for row, clock, load, label, legend_id in [
            (2, dates, NGCP, 'NGCP 1 AM', top_legend),
            (4, blocks, NGCP_4D, 'NGCP 4D', bottom_legend),
        ]:
            y = (
                load.reindex(clock)
                if view == 'radiance'
                else panel_z(load, left, right).reindex(clock)
            )
            fig.add_trace(
                go.Scatter(
                    x=clock,
                    y=y,
                    mode='lines',
                    name=label,
                    legend=legend_id,
                    line=dict(
                        color=COLOR['load'],
                        width=1.5,
                        shape='linear' if row == 2 else 'hv',
                    ),
                    connectgaps=False,
                ),
                row=row,
                col=col,
                secondary_y=(view == 'radiance'),
            )
            if view == 'radiance':
                fig.update_yaxes(
                    range=[0, PLOT_LIMIT[row]],
                    row=row,
                    col=col,
                    secondary_y=False,
                )
            fig.add_vline(
                x=HAIYAN,
                line_dash='dash',
                line_color='#233B60',
                row=row,
                col=col,
            )

        for row in range(1, 5):
            fig.update_xaxes(range=[left, right], row=row, col=col)
        fig.update_xaxes(showticklabels=True, row=2, col=col)
        fig.update_xaxes(title_text='Date', row=4, col=col)

    for row in (2, 4):
        fig.update_yaxes(
            title_text=(
                'Radiance (nW·cm⁻²·sr⁻¹)'
                if view == 'radiance' else 'z-score'
            ),
            row=row,
            col=1,
            secondary_y=False,
        )
        if view == 'radiance':
            fig.update_yaxes(
                title_text='NGCP load (MW)',
                row=row,
                col=2,
                secondary_y=True,
            )
        else:
            for col in (1, 2):
                fig.update_yaxes(
                    visible=False,
                    row=row,
                    col=col,
                    secondary_y=True,
                )

    legends = {
        'legend': dict(x=.53, y=.905),
        'legend2': dict(x=.99, y=.905),
        'legend3': dict(x=.53, y=.39),
        'legend4': dict(x=.99, y=.39),
    }
    for item in legends.values():
        item.update(
            orientation='h',
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255,255,255,.9)',
            font=dict(size=11),
        )

    fig.update_layout(
        template='plotly_white',
        paper_bgcolor='rgba(0,0,0,0)',
        width=1500,
        height=850,
        margin=dict(l=85, r=70, t=20, b=75),
        font=dict(size=12, color='#233B60'),
        **legends,
    )
    save_fig(fig, f'fig_01_2013_2023_clean_{view}')

clean_main_figure('radiance')
clean_main_figure('zscore')

In [ ]:
rows = []
periods = [
    ('2013–2023', FULL_LEFT, PLOT_RIGHT),
    ('Haiyan zoom', ZOOM_LEFT, ZOOM_RIGHT),
]
signals = [
    ('Daily', 'Direct', DIRECT, NGCP),
    ('Daily', 'Gap-Filled', FILLED, NGCP),
    ('4D blocks', 'Area 4D', DIRECT_4D, NGCP_4D),
    ('4D blocks', 'G3 4D', G3_4D, NGCP_4D),
    ('4D blocks', 'RQ G3 4D · SC≥10%', RQ_4D, NGCP_4D),
]

for period, left, right in periods:
    for unit, label, ntl, load in signals:
        eligible = load.loc[left:right].dropna().index
        m = paired_metrics(ntl, load, eligible)
        rows.append({
            'Period': period,
            'Signal': label,
            'Unit': unit,
            'r': m['r'],
            'RMSE (paired z)': m['rmse_z'],
            'N%': m['n_pct'],
            'n': m['n_pairs'],
            'eligible': m['n_possible'],
        })

SUMMARY = pd.DataFrame(rows)
SUMMARY.to_csv(OUT / 'fig_01_summary_metrics.csv', index=False)

shown = SUMMARY.copy()
shown['r'] = shown['r'].map(lambda x: f'{x:.2f}' if pd.notna(x) else '—')
shown['RMSE (paired z)'] = shown['RMSE (paired z)'].map(
    lambda x: f'{x:.2f}' if pd.notna(x) else '—'
)
shown['N%'] = shown['N%'].map(
    lambda x: f'{x:.1f}%' if pd.notna(x) else '—'
)
shown['n / eligible'] = (
    shown['n'].astype(str) + ' / ' + shown['eligible'].astype(str)
)
display(shown[
    ['Period', 'Signal', 'Unit', 'r',
     'RMSE (paired z)', 'N%', 'n / eligible']
])

**Interpretation gate.** Inspect direct support alongside apparent Gap-Filled flat stretches; the CSV alone does not establish the exact gap-filling mechanism, so do not claim every flat section literally repeats the last observation. The opening direct and Gap-Filled columns are paired from one export. The SC panel comes from the RQ1 regional direct-statistics export and should be checked for footprint consistency before using it as a pixel-level explanation of the companion CSV.


## 2. Bring in the long RQ1 direct record and GHSL masks

**Question.** Does the relationship change as the spatial extent narrows from the **study area without a GHSL mask** to settlement-focused support? RQ1 Notebook 00 provides the unmasked regional direct-band CSV; Notebook 03 provides eight cumulative GHSL-mask CSVs. The unmasked area's `NTL_mean` and GHSL masks' `Mean_by_sum` are compared as standardized trajectories and metrics, not absolute radiance levels. **G8 is not the unmasked study area:** G8 retains GHSL classes 10–30 and excludes pixels outside those classes. Their rows are placed together in the figures so the extent difference is visible.

Following Notebook 03, each direct series requires positive valid-pixel count and uses a **0th–95th percentile range** (upper 5% capped). `SC = 100 × Valid_px / max(Valid_px)` is normalized separately within each spatial support. A common τ therefore means the same fraction of that support's own maximum observed valid pixels, not an equal pixel count across supports. At τ=0, all dates with direct valid pixels remain eligible.


In [ ]:
def read_daily_csv(path, value_col):
    if not path.is_file():
        raise FileNotFoundError(path)
    d = date_from_frame(pd.read_csv(path))
    missing = {value_col,'Valid_px'} - set(d.columns)
    if missing:
        raise ValueError(f'{path.name}: missing {sorted(missing)}')
    d['value'] = pd.to_numeric(d[value_col],errors='coerce')
    d['Valid_px'] = pd.to_numeric(d.Valid_px,errors='coerce')
    return d[['value','Valid_px']].reindex(CALENDAR)

def prep_direct(d):
    f=d.copy()
    valid=(f.Valid_px>0)&f.value.notna()&np.isfinite(f.value)
    if not valid.any():
        raise ValueError('No direct dates in a spatial scale')
    denominator=float(f.loc[valid,'Valid_px'].max())
    hi=float(f.loc[valid,'value'].quantile(.95))
    f['SC_pct']=100*f.Valid_px/denominator
    f['ntl']=f.value.clip(upper=hi).where(valid)
    return f[['ntl','Valid_px','SC_pct']],{'max_valid_px':denominator,'upper_95_cap':hi,'direct_dates':int(valid.sum())}

REGION_DAILY=read_daily_csv(REGIONAL_CSV,'NTL_mean')
SCALES={FULL_LABEL:prep_direct(REGION_DAILY)[0]}
inventory=[{'scale':FULL_LABEL,'source':str(REGIONAL_CSV),'value_col':'NTL_mean',
            **prep_direct(REGION_DAILY)[1]}]
for group,(mask,label,color) in GROUPS.items():
    path=GHSL_DIR / f'EasternVisayas_DNBBRDF_{group}_stats.csv'
    d=read_daily_csv(path,'Mean_by_sum')
    SCALES[mask],audit=prep_direct(d)
    inventory.append({'scale':mask,'source':str(path),'value_col':'Mean_by_sum',**audit})
INPUT_AUDIT=pd.DataFrame(inventory)
INPUT_AUDIT.to_csv(OUT/'qualified_input_inventory.csv',index=False)
print(f'Loaded long direct series for {len(SCALES)} spatial scales; source and denominator audit saved.')

### Figure 2 — The spatial and observational qualification steps

The top panel plots Pearson $r$ for the unsmoothed P95-capped direct series over the **full SC threshold range**; the lower panel shows **N%**, the fraction of load dates with an admitted direct observation. Read each line as a trade-off, not a universal optimum. A GHSL mask changes which pixels contribute; τ changes which dates are admitted within that mask. Because each τ has a different paired-date set, a higher $r$ alone is insufficient evidence of better measurement.


In [ ]:
intro_rows=[]
for scale,d in SCALES.items():
    for tau in SC_SWEEP_TAUS:
        admitted=d.ntl.notna()&(d.SC_pct>=tau)&LOAD.notna()
        m=paired_metrics(d.ntl.where(admitted),LOAD,LOAD.index[LOAD.notna()])
        intro_rows.append({'scale':scale,'tau':tau,'direct_pct':100*admitted.sum()/LOAD.notna().sum(),**m})
INTRO=pd.DataFrame(intro_rows)
INTRO.to_csv(OUT/'reliability_conditions_direct_dates.csv',index=False)
FIG2_SCALES=MASK_ORDER
fig=make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.11,
                  subplot_titles=['Direct NTL–load alignment','Directly paired load dates (N%)'])
for scale in FIG2_SCALES:
    q=INTRO[INTRO.scale==scale].sort_values('tau')
    for row,field in [(1,'r'),(2,'direct_pct')]:
        fig.add_trace(go.Scatter(x=q.tau,y=q[field],mode='lines',name=scale,
                                 showlegend=(row==1),line=dict(color=COLOR[scale],width=3)),row=row,col=1)
for row in (1,2):
    for threshold in (10,50):
        fig.add_vline(x=threshold,row=row,col=1,line_dash='dot',line_color='#9CACBD')
fig.update_xaxes(title_text='SC threshold τ (%)',range=[0,100],dtick=10,row=2,col=1)
fig.update_yaxes(title_text='Pearson r',row=1,col=1)
fig.update_yaxes(title_text='N% of load dates',range=[0,100],row=2,col=1)
fig.update_layout(template='plotly_white',height=730,width=1200,
                  title='SC threshold trade-off after P95 capping: 2012–2024',
                  legend=dict(orientation='v',x=1.02,y=1,font=dict(size=13)),
                  paper_bgcolor='rgba(0,0,0,0)',margin=dict(l=75,r=270,t=90,b=75),
                  font=dict(color='#233B60',size=13))
save_fig(fig,'fig_02_extent_and_sc_introduction')


In [ ]:
INTRO = pd.DataFrame(intro_rows)
INTRO.to_csv(
    OUT / 'reliability_conditions_direct_dates_2013_2024.csv',
    index=False,
)

# Take the names directly from your existing RQ1 GROUPS definition.
GROUP_LABELS = {
    group_id: f'{group_id} {description}'
    for group_id, description, _ in GROUPS.values()
}
FIG2_SCALES = [f'G{i}' for i in range(1, 9)]
CHECKPOINTS = (10, 50, 90)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=.09,
    row_heights=[.52, .48],
)

for scale in FIG2_SCALES:
    q = INTRO[INTRO.scale == scale].sort_values('tau')
    checkpoints = q[q.tau.isin(CHECKPOINTS)]

    fig.add_trace(
        go.Scatter(
            x=q.tau,
            y=q.r,                 # no added 30-pair cutoff
            mode='lines',
            name=GROUP_LABELS[scale],
            line=dict(color=COLOR[scale], width=2.6),
        ),
        row=1,
        col=1,
    )

    # Checkpoint dots for every G1–G8 correlation curve.
    fig.add_trace(
        go.Scatter(
            x=checkpoints.tau,
            y=checkpoints.r,
            customdata=checkpoints.n_pairs,
            mode='markers',
            showlegend=False,
            marker=dict(
                color=COLOR[scale],
                size=7,
                line=dict(color='white', width=1),
            ),
            hovertemplate=(
                f'{GROUP_LABELS[scale]}<br>'
                'τ=%{x}%<br>r=%{y:.2f}<br>'
                'Paired dates=%{customdata}<extra></extra>'
            ),
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=q.tau,
            y=q.n_pct,
            mode='lines',
            name=GROUP_LABELS[scale],
            showlegend=False,
            line=dict(color=COLOR[scale], width=2.6),
        ),
        row=2,
        col=1,
    )

# The guides become darker as the SC threshold increases.
CHECKPOINT_COLORS = {
    10: '#C6D9DF',
    50: '#779EAA',
    90: '#315E6D',
}
for tau, guide_color in CHECKPOINT_COLORS.items():
    for row in (1, 2):
        fig.add_vline(
            x=tau,
            row=row,
            col=1,
            line_dash='dash',
            line_width=1.5,
            line_color=guide_color,
        )

# Label the three representative curves at each checkpoint.
# All eight curves still have checkpoint dots and full names in the legend.
BOTTOM_OFFSETS = {
    10: {'G1': 25, 'G5': -18, 'G8': -42},
    50: {'G1': 25, 'G5': -18, 'G8': -42},
    90: {'G1': -18, 'G5': -40, 'G8': -62},
}

for tau in CHECKPOINTS:
    for scale in ('G1', 'G5', 'G8'):
        point = INTRO[
            (INTRO.scale == scale) & (INTRO.tau == tau)
        ]
        if point.empty:
            continue

        r = float(point.r.iloc[0])
        retained = float(point.n_pct.iloc[0])

        if np.isfinite(r):
            fig.add_annotation(
                x=tau,
                y=r,
                text=f'{scale} {r:.2f}',
                showarrow=True,
                arrowhead=0,
                arrowwidth=1,
                arrowcolor=COLOR[scale],
                ax=0,
                ay=-19,
                bgcolor='rgba(255,255,255,.92)',
                bordercolor=COLOR[scale],
                borderwidth=.7,
                borderpad=2,
                font=dict(color=COLOR[scale], size=10),
                row=1,
                col=1,
            )

        fig.add_trace(
            go.Scatter(
                x=[tau],
                y=[retained],
                mode='markers',
                showlegend=False,
                marker=dict(
                    color=COLOR[scale],
                    size=8,
                    line=dict(color='white', width=1),
                ),
            ),
            row=2,
            col=1,
        )
        fig.add_annotation(
            x=tau,
            y=retained,
            text=f'{scale} {retained:.1f}%',
            showarrow=True,
            arrowhead=0,
            arrowwidth=1,
            arrowcolor=COLOR[scale],
            ax=0,
            ay=BOTTOM_OFFSETS[tau][scale],
            bgcolor='rgba(255,255,255,.92)',
            bordercolor=COLOR[scale],
            borderwidth=.7,
            borderpad=2,
            font=dict(color=COLOR[scale], size=10),
            row=2,
            col=1,
        )

fig.update_xaxes(
    title_text='SC threshold τ (%)',
    range=[0, 100],
    dtick=10,
    row=2,
    col=1,
)
fig.update_yaxes(title_text='Pearson r', row=1, col=1)
fig.update_yaxes(
    title_text='N% retained coverage',
    range=[0, 105],
    row=2,
    col=1,
)
fig.update_layout(
    template='plotly_white',
    width=1250,
    height=760,
    title='SC threshold trade-off after P95 capping, 2013–2024',
    legend=dict(
        orientation='v',
        x=1.02,
        y=1,
        font=dict(size=11),
    ),
    paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=75, r=265, t=80, b=75),
    font=dict(color='#233B60', size=13),
)
save_fig(fig, 'fig_02_sc_tradeoff_2013_2024')

Discussion: Alignment generally increased as the spatial-completeness threshold rose, while the fraction of retained load dates declined. Because thresholding selects dates rather than modifying their radiance, the increase represents stronger conditional agreement on well-observed dates; it does not establish better performance across the full calendar. Stringent thresholds may exclude precisely the cloud-affected storm periods of interest. We therefore evaluate correlation together with retained coverage and select thresholds by the intended application, rather than maximizing correlation alone.

## 3. Sensitivity cube: spatial scale × SC threshold × 1–30 calendar days

**Question.** What smoothing horizon is enough for a useful long-term signal, and how much support does it require? We compute **both** of the following lenses for every combination:

- **Direct-date lens (primary):** apply SC to daily direct NTL; smooth the admitted values over a centred *calendar* window; evaluate only dates with a directly admitted NTL observation and a load observation. `n%` is therefore normally constant across windows at fixed mask and τ. This makes the trade-off between $r$ and loss of directly observed dates explicit.
- **Window-supported lens (diagnostic):** require at least 50% of the calendar days in each window to contain an SC-admitted direct observation. An estimate can then exist on a date without a direct NTL observation. Its `n%` is **supported-estimate availability**, not directly observed date coverage. We report the median number of contributing direct days and show 25/50/75% support-rule sensitivity later.

The primary retrospective windows are centred, matching the spirit of Notebook 03's retrospective smoothing. A trailing-window check follows for prospective use. Both NTL and NGCP load are smoothed on the same calendar width; consequently rising $r$ describes agreement at a broader time scale, not better daily outage detection. `w=1` is unsmoothed direct NTL. This calendar-day sweep is **not** a reproduction of the original 30-retained-observation method.

In [ ]:
def smooth_pair(d,load,tau,w,lens='direct',support_fraction=.50,center=True):
    admitted=d.ntl.notna()&(d.Valid_px>0)&(d.SC_pct>=tau)
    x=d.ntl.where(admitted)
    if lens=='complete':
        # Trailing, nested windows: every constituent calendar day must be direct and SC-admitted.
        # This guarantees N%(w+1) <= N%(w), unlike partial windows that can bridge gaps.
        count=x.notna().astype(int).rolling(w,min_periods=w).sum()
        xs=x.rolling(w,min_periods=w).mean()
        ys=load.rolling(w,min_periods=w).mean()
    elif lens in ('direct','window'):
        count=x.notna().astype(int).rolling(w,center=center,min_periods=1).sum()
        needed=1 if lens=='direct' else max(1,math.ceil(support_fraction*w))
        xs=x.rolling(w,center=center,min_periods=needed).mean()
        ys=load.rolling(w,center=center,min_periods=needed).mean()
        if lens=='direct': xs=xs.where(admitted)
    else:
        raise ValueError(f'Unknown support lens: {lens}')
    xs=xs.where(load.notna())
    ys=ys.where(load.notna())
    return xs,ys,admitted,count

def configuration_metric(d,load,tau,w,lens='direct',support_fraction=.50,center=True):
    x,y,admitted,count=smooth_pair(d,load,tau,w,lens,support_fraction,center)
    m=paired_metrics(x,y,load.index[load.notna()])
    paired=x.notna()&y.notna()
    m.update(direct_n=int((admitted&load.notna()).sum()),
             direct_pct=100*(admitted&load.notna()).sum()/load.notna().sum(),
             median_contributing_days=float(count[paired].median()) if paired.any() else np.nan,
             supported_without_target_direct=int((paired&~admitted).sum()))
    return m

# Synthetic logic checks only; these are not the RQ1 results.
toy_i=pd.date_range('2020-01-01',periods=5)
toy_d=pd.DataFrame({'ntl':[1.,np.nan,np.nan,4.,5.],
                    'Valid_px':[10,0,0,10,10],
                    'SC_pct':[100,0,0,100,100]},index=toy_i)
toy_l=pd.Series([1.,2.,3.,4.,5.],index=toy_i)
a,b,ad,c=smooth_pair(toy_d,toy_l,0,3,'direct')
assert a.isna().iloc[1] and ad.sum()==3
a,b,ad,c=smooth_pair(toy_d,toy_l,0,3,'window',.5)
assert c.iloc[3]>=2 and a.iloc[3]==a.iloc[3]
assert configuration_metric(toy_d,toy_l,0,3,'complete')['n_pairs'] <= configuration_metric(toy_d,toy_l,0,1,'complete')['n_pairs']

rows=[]
for scale,d in SCALES.items():
    for tau in TAUS:
        for w in WINDOWS:
            for lens in ['direct','window','complete']:
                rows.append({'scale':scale,'tau':tau,'window_days':w,'lens':lens,
                             'window_rule':'trailing complete' if lens=='complete' else 'centered partial',
                             **configuration_metric(d,LOAD,tau,w,lens,MIN_WINDOW_FRACTION)})
MATRIX=pd.DataFrame(rows)
MATRIX.to_csv(OUT/'mask_sc_window_1_to_30_metric_cube.csv',index=False)
check=MATRIX[MATRIX.lens=='complete'].sort_values('window_days').groupby(['scale','tau']).n_pairs.apply(lambda s:s.diff().dropna().le(0).all())
assert check.all(), 'Complete-window paired N must not increase with width'
print(f'Saved {len(MATRIX):,} mask × SC × window × support-rule results.')


### Figure 3 — 1–30-day sensitivity on the same paired-date rule

Each row now uses **complete trailing windows**. At width $w$, the endpoint is scored only if **all $w$ calendar days** have an SC-admitted direct observation and NGCP load. Thus $r$, paired z-score RMSE and **N%** are computed on the same window-qualified endpoints; N% must fall or stay level as $w$ grows. Missing curve segments mean too few valid pairs to estimate $r$, an important limitation rather than a plotting error. This strict rule is deliberately conservative in a cloud-impacted setting.

The exported cube also includes the previous **direct-anchor** rule (center date must be observed) and a **50%-supported partial-window** rule. Their N% can stay flat or rise, respectively; those are different estimands and are shown separately in the supplement. RMSE is mathematically linked to $r$ after paired standardization: $\mathrm{RMSE}^2=2(1-r)$.


In [ ]:
WINDOW_CHECKPOINTS = (4, 7, 14, 30)
ALL_GROUPS = [f'G{i}' for i in range(1, 9)]
GROUP_LABELS = {
    group_id: f'{group_id} {description}'
    for group_id, description, _ in GROUPS.values()
}
METRIC_LABELS = {
    'r': 'Pearson r',
    'rmse_z': 'Paired z-score RMSE',
    'n_pct': 'N% retained coverage',
}

def metric_curve_figure(df, metric, scales, taus, lens, name):
    fig = make_subplots(
        rows=1,
        cols=3,
        shared_yaxes=True,
        horizontal_spacing=.055,
        subplot_titles=[f'SC τ={tau}%' for tau in taus],
    )

    x_offset = {4: 18, 7: 32, 14: 0, 30: -32}
    y_offset = {
        4:  {'G1': -18, 'G5': 18,  'G8': 38},
        7:  {'G1': 18,  'G5': -18, 'G8': -38},
        14: {'G1': -18, 'G5': 18,  'G8': 38},
        30: {'G1': 18,  'G5': -18, 'G8': -38},
    }

    for col, tau in enumerate(taus, start=1):
        panel = df[(df.lens == lens) & (df.tau == tau)]

        for day in WINDOW_CHECKPOINTS:
            fig.add_vline(
                x=day,
                row=1,
                col=col,
                line_dash='dot',
                line_color='#B6C7D3',
                line_width=1,
            )

        for scale in scales:
            q = panel[panel.scale == scale].sort_values('window_days')

            fig.add_trace(
                go.Scatter(
                    x=q.window_days,
                    y=q[metric],
                    mode='lines',
                    name=GROUP_LABELS[scale],
                    showlegend=(col == 1),
                    line=dict(color=COLOR[scale], width=2.2),
                    connectgaps=False,
                ),
                row=1,
                col=col,
            )

        # Match the SC plot's small outlined labels on selected curves.
        for day in WINDOW_CHECKPOINTS:
            for scale in ('G1', 'G5', 'G8'):
                point = panel[
                    (panel.scale == scale) &
                    (panel.window_days == day)
                ]
                if point.empty:
                    continue

                value = point[metric].iloc[0]
                if pd.isna(value):
                    continue

                label = (
                    f'{scale} {value:.1f}%'
                    if metric == 'n_pct'
                    else f'{scale} {value:.2f}'
                )
                ay = y_offset[day][scale]

                # Keep low-coverage labels above the axis.
                if metric == 'n_pct' and value < 10:
                    ay = {'G1': -18, 'G5': -38, 'G8': -58}[scale]

                fig.add_trace(
                    go.Scatter(
                        x=[day],
                        y=[value],
                        mode='markers',
                        showlegend=False,
                        marker=dict(
                            color=COLOR[scale],
                            size=6,
                            line=dict(color='white', width=1),
                        ),
                    ),
                    row=1,
                    col=col,
                )
                fig.add_annotation(
                    x=day,
                    y=value,
                    text=label,
                    showarrow=True,
                    arrowhead=0,
                    arrowwidth=.8,
                    arrowcolor=COLOR[scale],
                    ax=x_offset[day],
                    ay=ay,
                    bgcolor='rgba(255,255,255,.95)',
                    bordercolor=COLOR[scale],
                    borderwidth=.7,
                    borderpad=2,
                    font=dict(color=COLOR[scale], size=9),
                    row=1,
                    col=col,
                )

        fig.update_xaxes(
            title_text='Calendar window (days)',
            range=[1, 30.5],
            tickvals=[1, 4, 7, 14, 20, 25, 30],
            row=1,
            col=col,
        )

    fig.update_yaxes(
        title_text=METRIC_LABELS[metric],
        row=1,
        col=1,
    )
    if metric == 'n_pct':
        fig.update_yaxes(range=[0, 100])

    fig.update_layout(
        template='plotly_white',
        width=1650,
        height=580,
        title=name,
        legend=dict(x=1.01, y=1, font=dict(size=11)),
        paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=80, r=245, t=85, b=85),
        font=dict(color='#233B60', size=12),
    )
    return fig

In [ ]:
save_fig(
    metric_curve_figure(
        MATRIX, 'r', ALL_GROUPS, [0, 10, 50], 'complete',
        'Alignment across calendar windows and SC thresholds',
    ),
    'fig_03a_r_window_curves',
)

In [ ]:
save_fig(
    metric_curve_figure(
        MATRIX, 'rmse_z', ALL_GROUPS, [0, 10, 50], 'complete',
        'Paired standardized error across calendar windows and SC thresholds',
    ),
    'fig_03b_rmse_window_curves',
)

In [ ]:
save_fig(
    metric_curve_figure(
        MATRIX, 'n_pct', ALL_GROUPS, [0, 10, 50], 'complete',
        'Retained coverage across calendar windows and SC thresholds',
    ),
    'fig_03c_n_window_curves',
)

### Figure 4 — SC and temporal support are separate dimensions

**Figure 4A** colors direct **N%** by spatial support and SC threshold. **Figures 4B–C** put 1–30 days on the horizontal axis; their color is complete-window **N%** or $r$ on the *same eligible endpoints*. G8 sits beside the unmasked study area, making their difference easy to inspect. A 50%-supported window heatmap is retained as a supplement to show how partial windows can create more estimates when widened; those estimates are not complete direct observation. SC is a spatial fraction, normalized within each mask, and does not itself vary with temporal width.


In [ ]:
direct_heat=INTRO.pivot(index='scale',columns='tau',values='direct_pct').reindex(MASK_ORDER)
fig=go.Figure(go.Heatmap(x=direct_heat.columns,y=direct_heat.index,z=direct_heat.values,
                         colorscale='Blues',zmin=0,zmax=100,colorbar=dict(title='N%'),
                         hovertemplate='Scale %{y}<br>SC τ=%{x}%<br>Direct dates=%{z:.1f}%<extra></extra>'))
fig.update_layout(template='plotly_white',width=1080,height=560,
                  title='Direct N% by spatial support and SC threshold',
                  xaxis_title='SC threshold τ (%)',yaxis_title='Spatial scale',
                  margin=dict(l=190,r=35,t=75,b=70),font=dict(color='#233B60'))
save_fig(fig,'fig_04a_direct_retention_sc_heatmap')

def window_heatmap(metric,lens,taus,name,title):
    f=make_subplots(rows=1,cols=len(taus),subplot_titles=[f'SC τ={t}%' for t in taus],
                    shared_yaxes=True,horizontal_spacing=.035)
    for col,tau in enumerate(taus,start=1):
        q=MATRIX[(MATRIX.lens==lens)&(MATRIX.tau==tau)]
        p=q.pivot(index='scale',columns='window_days',values=metric).reindex(MASK_ORDER)
        f.add_trace(go.Heatmap(x=p.columns,y=p.index,z=p.values,colorscale='Blues',
                               zmin=0 if metric=='n_pct' else float(MATRIX[(MATRIX.lens==lens)&(MATRIX.tau.isin(taus))][metric].min()),
                               zmax=100 if metric=='n_pct' else 1,
                               showscale=(col==len(taus)),
                               colorbar=dict(title='N%' if metric=='n_pct' else 'Pearson r'),
                               hovertemplate='Scale %{y}<br>Window %{x} d<br>Value %{z:.2f}<extra></extra>'),row=1,col=col)
    f.update_xaxes(title_text='Calendar window (days)',dtick=5)
    f.update_yaxes(title_text='Spatial scale',row=1,col=1)
    f.update_layout(template='plotly_white',width=1450,height=570,title=title,
                    margin=dict(l=190,r=75,t=75,b=75),font=dict(color='#233B60'))
    save_fig(f,name)

window_heatmap('n_pct','complete',[0,10,50],'fig_04b_complete_n_mask_window_heatmap',
               'Complete-window N%: spatial support × width, by SC threshold')
window_heatmap('r','complete',[0,10,50],'fig_04c_complete_r_mask_window_heatmap',
               'Complete-window r: same settings and paired endpoints')
window_heatmap('n_pct','window',[0,10,50],'fig_s_partial_window_n_heatmap',
               'Supplement: N% with at least 50% direct support in each window')

## 4. Choosing a window and testing composite-score weights

**Step 1: preserve temporal resolution.** For each GHSL mask × τ, the notebook selects the **shortest** 1–30-day direct-anchor window within Δ$r=0.02$ of that setting's best $r$. The result is repeated at Δ$r=0.01$ and 0.05. This is a declared practical tolerance, not a significance test. The direct-anchor score is the manuscript-comparable branch: its **N% reflects admitted direct dates and does not depend on window width**; width is handled by the near-plateau rule. The complete-window curves above quantify the stricter temporal-support cost.

**Step 2: make preferences explicit.** The primary two-axis score varies alignment weight $\lambda$ from 0 to 1:

$$S_\lambda=\lambda(r+1)/2+(1-\lambda)N_{\rm direct}/100.$$

This asks how much selection changes when an application values load alignment over directly observed temporal coverage. Paired standardized RMSE is linked exactly to $r$ by $\mathrm{RMSE}^2=2(1-r)$, so counting both as independent evidence would double-weight alignment.

**Step 3: stress-test the manuscript's three weights.** We also sweep **all 231 weight triples** in 0.05 increments that sum to one across $r$, inverse paired-z RMSE, and N%. Fixed 0–1 scales are used: $(r+1)/2$, $1-\mathrm{RMSE}/2$, and $N/100$. The manuscript's **0.6/0.1/0.3** point is marked. The fixed scaling makes this a sensitivity audit of those weights, not a numerical reproduction of Table 4's min–max normalization. The winner map and top-five frequency show whether that choice sits within a stable region. Since $r$ and RMSE are redundant, this three-weight sweep is a *robustness audit of the historical score*, while the two-axis score is the cleaner decision rule.


In [ ]:
def plateau_candidates(frame,delta,group_keys=['scale','tau']):
    candidates=[]
    source=frame[(frame.lens=='direct')&frame.r.notna()&
                 (frame.n_pct>=100*MIN_DIRECT_COVERAGE)&(frame.n_pairs>=30)].copy()
    for _,g in source.groupby(group_keys,sort=False):
        best=g.r.max()
        pick=g[g.r>=best-delta].sort_values('window_days').iloc[0]
        candidates.append(pick)
    return pd.DataFrame(candidates).reset_index(drop=True) if candidates else pd.DataFrame()

LOCAL_CANDIDATES=plateau_candidates(MATRIX,PLATEAU_DELTA_R)
LOCAL_CANDIDATES=LOCAL_CANDIDATES[LOCAL_CANDIDATES.scale!=FULL_LABEL].copy()
if LOCAL_CANDIDATES.empty:
    raise ValueError('No locally eligible GHSL candidates: inspect n, SC and minimum coverage')
LOCAL_CANDIDATES.to_csv(OUT/'local_shortest_near_plateau_candidates.csv',index=False)

def primary_score(frame,lam):
    q=frame.copy()
    q['score']=lam*(q.r+1)/2+(1-lam)*q.n_pct/100
    return q.sort_values(['score','n_pct','window_days','scale','tau'],
                         ascending=[False,False,True,True,True]).reset_index(drop=True)

LOCAL_WEIGHTS=[]
for lam in np.round(np.arange(0,1.0001,.05),2):
    winner=primary_score(LOCAL_CANDIDATES,lam).iloc[0]
    LOCAL_WEIGHTS.append({'lambda_alignment':float(lam),'selected_scale':winner.scale,
                          'selected_tau':int(winner.tau),'selected_window_days':int(winner.window_days),
                          'r':winner.r,'rmse_z':winner.rmse_z,'n_pairs':int(winner.n_pairs),
                          'n_pct':winner.n_pct,'score':winner.score})
LOCAL_WEIGHTS=pd.DataFrame(LOCAL_WEIGHTS)
LOCAL_WEIGHTS.to_csv(OUT/'local_weight_sweep_selected_configurations.csv',index=False)

def minmax(s):
    span=s.max()-s.min()
    return pd.Series(.5,index=s.index) if not np.isfinite(span) or span==0 else (s-s.min())/span
historic=LOCAL_CANDIDATES.copy()
historic['historical_score_on_new_grid']=(.6*minmax(historic.r)
                                          +.1*minmax(-historic.rmse_z)
                                          +.3*minmax(historic.n_pct))
historic=historic.sort_values('historical_score_on_new_grid',ascending=False)
historic.to_csv(OUT/'historical_weights_on_new_grid_NOT_Table4.csv',index=False)
print('Original min-max 0.6/0.1/0.3 form on the new candidate grid (not Table 4):',
      historic.iloc[0][['scale','tau','window_days','r','n_pct']].to_dict())
print('Maximum paired-z identity discrepancy:',
      float(np.nanmax(abs(LOCAL_CANDIDATES.rmse_z**2-2*(1-LOCAL_CANDIDATES.r)))))

plateau_rows=[]
for delta in [.01,.02,.05]:
    q=plateau_candidates(MATRIX,delta)
    q=q[q.scale.isin(['G1','G2','G3'])]
    for lam in [.3,.5,.7]:
        if len(q):
            x=primary_score(q,lam).iloc[0]
            plateau_rows.append({'delta_r':delta,'lambda_alignment':lam,'scale':x.scale,
                                 'tau':x.tau,'window_days':x.window_days,'r':x.r,'n_pct':x.n_pct})
PLATEAU_CHECK=pd.DataFrame(plateau_rows)
PLATEAU_CHECK.to_csv(OUT/'window_plateau_tolerance_sensitivity.csv',index=False)
# Explicit 231-point simplex sweep of the manuscript's three score weights.
# All axes use fixed theoretical scales; candidate-set min/max cannot move the goalposts.
score_candidates=LOCAL_CANDIDATES.copy()
score_candidates['setting']=(score_candidates.scale+' / τ='+score_candidates.tau.astype(int).astype(str)
                             +'% / '+score_candidates.window_days.astype(int).astype(str)+' d')
score_candidates['r01']=(score_candidates.r+1)/2
score_candidates['error01']=1-score_candidates.rmse_z/2
score_candidates['n01']=score_candidates.n_pct/100
simplex_rows=[]
top5_count={s:0 for s in score_candidates.setting}
winner_count={s:0 for s in score_candidates.setting}
for ar in range(21):
    for ae in range(21-ar):
        an=20-ar-ae
        wr,we,wn=ar/20,ae/20,an/20
        q=score_candidates.copy()
        q['score3']=wr*q.r01+we*q.error01+wn*q.n01
        q=q.sort_values(['score3','n_pct','window_days','setting'],
                        ascending=[False,False,True,True])
        winner=q.iloc[0]
        winner_count[winner.setting]+=1
        for setting in q.head(5).setting:
            top5_count[setting]+=1
        simplex_rows.append({'w_r':wr,'w_error':we,'w_n':wn,
                             'selected_setting':winner.setting,'selected_scale':winner.scale,
                             'selected_tau':int(winner.tau),'selected_window_days':int(winner.window_days),
                             'selected_r':winner.r,'selected_rmse_z':winner.rmse_z,
                             'selected_n_pct':winner.n_pct,'score':winner.score3})
WEIGHT_SIMPLEX=pd.DataFrame(simplex_rows)
assert len(WEIGHT_SIMPLEX)==231
WEIGHT_SIMPLEX.to_csv(OUT/'composite_score_231_weight_simplex.csv',index=False)
WEIGHT_FREQUENCY=score_candidates[['setting','scale','tau','window_days','r','rmse_z','n_pct']].copy()
WEIGHT_FREQUENCY['winner_share']=WEIGHT_FREQUENCY.setting.map(winner_count)/len(WEIGHT_SIMPLEX)
WEIGHT_FREQUENCY['top_five_share']=WEIGHT_FREQUENCY.setting.map(top5_count)/len(WEIGHT_SIMPLEX)
WEIGHT_FREQUENCY=WEIGHT_FREQUENCY.sort_values(['top_five_share','winner_share'],ascending=False)
WEIGHT_FREQUENCY.to_csv(OUT/'composite_score_weight_stability.csv',index=False)
HISTORICAL_SELECTION=WEIGHT_SIMPLEX[(WEIGHT_SIMPLEX.w_r==.6)&
                                    (WEIGHT_SIMPLEX.w_error==.1)&
                                    (WEIGHT_SIMPLEX.w_n==.3)].iloc[0]
print('Fixed-scale 231-weight audit; 0.6/0.1/0.3 selects:',HISTORICAL_SELECTION.selected_setting)


### Figure 5 — The candidate frontier and score-weight sensitivity

The frontier shows $r$ against direct **N%** for each shortest-near-plateau setting. Figure 5B shows how the two-axis winner changes over the full $\lambda$ range. Figure 5C maps all **231 historical-style weight triples** on a simplex, with 0.6/0.1/0.3 highlighted. Figure 5D reports each setting's share of wins and top-five placements. A broad stable region supports a robust *selection rule*; rapid switching means the manuscript should report a family of application-dependent choices rather than one supposedly universal weight vector. These charts select against local NGCP; Section 5 checks held-out transfer.


In [ ]:
fig=go.Figure()
for scale in [v[0] for v in GROUPS.values()]:
    q=LOCAL_CANDIDATES[LOCAL_CANDIDATES.scale==scale]
    if q.empty: continue
    fig.add_trace(go.Scatter(x=q.n_pct,y=q.r,mode='markers',name=scale,
                             marker=dict(size=9,color=COLOR[scale],opacity=.72),
                             customdata=np.stack([q.tau,q.window_days],axis=1),
                             hovertemplate=f'{scale}<br>SC τ=%{{customdata[0]}}%<br>Window=%{{customdata[1]}} d<br>Direct N=%{{x:.1f}}%<br>r=%{{y:.3f}}<extra></extra>'))
fig.update_layout(template='plotly_white',width=1030,height=570,
                  title='Candidate frontier after the shortest-near-plateau window rule',
                  xaxis_title='Direct N% of load dates',yaxis_title='Pearson r',
                  legend=dict(orientation='h',y=-.21),margin=dict(l=70,r=25,t=75,b=115),font=dict(color='#233B60'))
save_fig(fig,'fig_05a_alignment_coverage_candidate_frontier')

fig=make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.10,
                  subplot_titles=['Selected SC threshold and mask','Alignment and direct-date coverage of selection'])
for scale in [v[0] for v in GROUPS.values()]:
    q=LOCAL_WEIGHTS[LOCAL_WEIGHTS.selected_scale==scale]
    if q.empty: continue
    fig.add_trace(go.Scatter(x=q.lambda_alignment,y=q.selected_tau,mode='markers',name=scale,
                             marker=dict(size=10,color=COLOR[scale]),
                             text=[f'{w} d' for w in q.selected_window_days],
                             hovertemplate='Alignment weight %{x:.2f}<br>SC τ=%{y}%<br>Window %{text}<extra>'+scale+'</extra>'),row=1,col=1)
fig.add_trace(go.Scatter(x=LOCAL_WEIGHTS.lambda_alignment,y=LOCAL_WEIGHTS.r,mode='lines+markers',
                         name='Selected r',line=dict(color='#4E79A7',width=2)),row=2,col=1)
fig.add_trace(go.Scatter(x=LOCAL_WEIGHTS.lambda_alignment,y=LOCAL_WEIGHTS.n_pct/100,
                         mode='lines+markers',name='Selected direct fraction',
                         line=dict(color='#375623',width=2)),row=2,col=1)
fig.update_xaxes(title_text='Declared alignment priority λ',row=2,col=1)
fig.update_yaxes(title_text='SC threshold τ (%)',row=1,col=1)
fig.update_yaxes(title_text='r / fraction',row=2,col=1)
fig.update_layout(template='plotly_white',width=1110,height=660,
                  title='What changes when the application values alignment more than temporal coverage?',
                  legend=dict(orientation='h',y=-.13),margin=dict(l=75,r=25,t=80,b=110),font=dict(color='#233B60'))
save_fig(fig,'fig_05b_local_weight_sensitivity')
# 231-combination historical-style weight sensitivity, using the same palette as the RQ1 masks.
fig=go.Figure()
for scale in [v[0] for v in GROUPS.values()]:
    q=WEIGHT_SIMPLEX[WEIGHT_SIMPLEX.selected_scale==scale]
    if q.empty: continue
    fig.add_trace(go.Scatterternary(a=q.w_r,b=q.w_error,c=q.w_n,mode='markers',name=scale,
                                    marker=dict(size=8,color=COLOR[scale],opacity=.8),
                                    text=q.selected_setting,
                                    hovertemplate='r weight %{a:.2f}<br>Error weight %{b:.2f}<br>N% weight %{c:.2f}<br>Winner %{text}<extra></extra>'))
fig.add_trace(go.Scatterternary(a=[.6],b=[.1],c=[.3],mode='markers',name='Manuscript 0.6/0.1/0.3',
                                marker=dict(symbol='star',size=17,color='#D65A4A',line=dict(color='#233B60',width=1)),
                                text=[HISTORICAL_SELECTION.selected_setting],
                                hovertemplate='Manuscript weights<br>Winner %{text}<extra></extra>'))
fig.update_layout(template='plotly_white',width=930,height=740,
                  title='Composite score: winning GHSL setting across 231 weight triples',
                  ternary=dict(sum=1,aaxis=dict(title='r weight'),
                               baxis=dict(title='inverse-RMSE weight'),
                               caxis=dict(title='N% weight')),
                  legend=dict(orientation='h',y=-.06),margin=dict(l=55,r=55,t=80,b=115),
                  font=dict(color='#233B60',size=13))
save_fig(fig,'fig_05c_composite_score_231_weight_simplex')

q=WEIGHT_FREQUENCY.head(12).sort_values('top_five_share')
fig=go.Figure()
fig.add_trace(go.Bar(x=q.top_five_share,y=q.setting,orientation='h',name='Top-five share',
                     marker_color='#4E79A7'))
fig.add_trace(go.Scatter(x=q.winner_share,y=q.setting,mode='markers',name='Winner share',
                         marker=dict(color='#D65A4A',size=10)))
fig.update_layout(template='plotly_white',width=1190,height=660,
                  title='Configuration stability across 231 composite-score weights',
                  xaxis_title='Share of weight combinations',xaxis_range=[0,1],
                  legend=dict(orientation='h',y=-.12),margin=dict(l=265,r=30,t=80,b=100),
                  font=dict(color='#233B60',size=13))
save_fig(fig,'fig_05d_composite_score_weight_stability')


## 5. Does the selection rule transfer when NGCP is unavailable locally?

A score containing $r$ or RMSE **cannot be recomputed in a target region without NGCP**. The testable alternative is to learn a fixed mask × SC × window policy on other island grids, apply it unchanged to a held-out grid, then reveal that grid's NGCP only for evaluation. This uses the RQ1 Notebook 04 CSVs and workbook; the loader audits the actual Hour 1 column selected on each sheet, including Negros. The intended design is five leave-one-grid-out folds. If a sheet cannot be safely parsed, the notebook reports it and runs the remaining eligible folds, without silently substituting a different load hour.

For each held-out grid and declared two-axis weight, the code chooses a shortest near-plateau window and ranks common candidates using training grids only. It compares held-out $r$ and N% against G8 τ=0, one day, and a clearly labelled *hindsight* local selection. The G8 comparator is GHSL classes 10–30, not the unmasked study area. Its values have the same P95 upper cap. A transfer result supports the **selection procedure**, subject to the number of valid folds; it cannot make load-based score components available where NGCP is absent.


In [ ]:
SHEET_MAP={
    'Panay':'PANAY HOURLY LOAD 2013-2024',
    'Negros':'NEGROS HOURLY LOAD 2013-2024',
    'Samar-Leyte':'LEY-SAM HOURLY LOAD 2013-2024',
    'Cebu':'CEBU HOURLY LOAD 2013-2024',
    'Bohol':'BOHOL LOAD 2013-2024',
}
REGION_NAMES={'SamarLeyte':'Samar-Leyte','BoholProvince':'Bohol',
              'CebuProvince':'Cebu','NegrosIsland':'Negros','PanayIsland':'Panay'}

def is_hour_one(label):
    s=re.sub(r'\s+','',str(label).strip().lower()).replace('a.m.','am')
    return bool(re.fullmatch(r'(?:hour|h)?0?1(?:\.0)?|0?1(?::00){1,2}(?:am)?|0?1am',s))

def read_ngcp_hour_one(workbook,sheet):
    diagnostics=[]
    for header in (1,0,2,3):
        f=pd.read_excel(workbook,sheet_name=sheet,header=header)
        f.columns=[str(x).strip() for x in f.columns]
        dates=pd.to_datetime(f.iloc[:,0],errors='coerce').dt.normalize()
        if dates.notna().sum()<30:
            diagnostics.append((header,list(f.columns[:8]),'fewer than 30 dates'))
            continue
        hour_col=next((c for c in f.columns[1:] if is_hour_one(c)),None)
        if hour_col is not None:
            values=pd.to_numeric(f[hour_col],errors='coerce')
            if values.notna().sum()>=30:
                return f,dates,hour_col,header,'explicit Hour 1 label'
        diagnostics.append((header,list(f.columns[:8]),'no explicit Hour 1 label'))
    # RQ1 Notebook 04 uses this position fallback. Keep it visible and audited.
    f=pd.read_excel(workbook,sheet_name=sheet,header=1)
    f.columns=[str(x).strip() for x in f.columns]
    dates=pd.to_datetime(f.iloc[:,0],errors='coerce').dt.normalize()
    numeric_like=[c for c in f.columns[1:] if re.fullmatch(r'\d+(?:\.0)?',str(c))]
    candidates=numeric_like or list(f.columns[1:])
    if dates.notna().sum()>=30 and candidates:
        hour_col=candidates[0]
        if pd.to_numeric(f[hour_col],errors='coerce').notna().sum()>=30:
            return f,dates,hour_col,1,'Notebook 04 positional fallback — verify header'
    raise ValueError(f'{sheet}: Hour 1 could not be identified. Header diagnostics: {diagnostics}')

def cross_region_inputs(folder,workbook):
    if not folder.is_dir() or not workbook.is_file():
        return {},{},f'Missing RQ1 Notebook 04 data: {folder} or {workbook}'
    files=sorted(folder.glob('*_DNBBRDF_class*_stats.csv'))
    if not files:
        return {},{},f'No Notebook 04 mask CSVs found in {folder}'
    region_data={}
    for path in files:
        parts=path.name.replace('_stats.csv','').split('_DNBBRDF_class')
        if len(parts)!=2 or parts[0] not in REGION_NAMES: continue
        region=REGION_NAMES[parts[0]]
        group='30' if parts[1]=='30-only' else parts[1]
        if group not in GROUPS: continue
        region_data[(region,GROUPS[group][0])]=prep_direct(read_daily_csv(path,'Mean_by_sum'))[0]
    loads={}; audit=[]; skipped=[]
    for region,sheet in SHEET_MAP.items():
        try:
            f,dates,hour_col,header,method=read_ngcp_hour_one(workbook,sheet)
            series=pd.Series(pd.to_numeric(f[hour_col],errors='coerce').to_numpy(),index=dates)
            series=series[series.index.notna()].dropna()
            if series.index.duplicated().any(): raise ValueError('duplicate dates')
            loads[region]=series.reindex(CALENDAR)
            audit.append({'region':region,'sheet':sheet,'hour_column':str(hour_col),
                          'excel_header_row':header+1,'selection_method':method,
                          'n_load_dates':int(loads[region].notna().sum())})
            print(f'{region}: Hour 1 = {hour_col!r}, Excel header row {header+1} ({method})')
        except (ValueError,KeyError) as exc:
            skipped.append(f'{region}: {exc}')
    pd.DataFrame(audit).to_csv(OUT/'cross_island_ngcp_hour1_column_audit.csv',index=False)
    for message in skipped: print('NGCP parsing:',message)
    complete=[region for region in loads if all((region,g) in region_data for g in [v[0] for v in GROUPS.values()])]
    if len(complete)<3:
        return {},{},f'Only {len(complete)} island grids have both Hour 1 load and all GHSL masks; at least three are needed.'
    return ({key:value for key,value in region_data.items() if key[0] in complete},
            {region:loads[region] for region in complete},'')

CROSS_DATA,CROSS_LOAD,CROSS_STATUS=cross_region_inputs(CROSS_DIR,NGCP_XLSX)
print(CROSS_STATUS or f'Loaded {len(CROSS_DATA)} region × mask direct inputs and {len(CROSS_LOAD)} NGCP grids.')


In [ ]:
CROSS_MATRIX=pd.DataFrame()
if not CROSS_STATUS:
    cross_rows=[]
    for (region,scale),d in CROSS_DATA.items():
        load=CROSS_LOAD[region]
        for tau in TRANSFER_TAUS:
            for w in WINDOWS:
                cross_rows.append({'region':region,'scale':scale,'tau':tau,'window_days':w,
                                   'lens':'direct',**configuration_metric(d,load,tau,w,'direct')})
    CROSS_MATRIX=pd.DataFrame(cross_rows)
    CROSS_MATRIX.to_csv(OUT/'cross_island_mask_sc_window_metric_cube.csv',index=False)
    print(f'Saved {len(CROSS_MATRIX):,} direct-date cross-island candidates.')
else:
    (OUT/'cross_island_status.txt').write_text(CROSS_STATUS+'\n')

# Optional supplementary comparison using only the direct summaries available for all islands.
# A 4-day rolling mean of regional CSV values is NOT the pixel-first Roman-inspired block composite.
if not CROSS_MATRIX.empty:
    comparison=CROSS_MATRIX[(CROSS_MATRIX.tau==0)&(CROSS_MATRIX.window_days.isin([1,4]))].copy()
    comparison['method']=np.where(comparison.window_days==1,'Direct daily regional summary',
                                  '4-day calendar mean of regional summaries')
    comparison[['region','scale','method','r','rmse_z','n_pairs','n_pct']].to_csv(
        OUT/'supplement_cross_island_direct_summary_comparison_NO_gapfilled.csv',index=False)

In [ ]:
def transfer_fold(matrix,holdout,lam,delta,region_data=None,region_load=None):
    train=matrix[matrix.region!=holdout].copy()
    train=train[train.r.notna()&(train.n_pairs>=30)]
    target=matrix[matrix.region==holdout].copy()
    regions=set(train.region.unique())
    n_training=len(set(matrix.region.unique()))-1
    if len(regions)!=n_training or n_training<2:
        raise ValueError(f'{holdout}: expected {n_training} training island grids')
    # A candidate must be present and sufficiently observed in every training region.
    agg=(train.groupby(['scale','tau','window_days'],as_index=False)
         .agg(median_r=('r','median'),median_pct=('n_pct','median'),
              min_pct=('n_pct','min'),regions=('region','nunique')))
    agg=agg[(agg.regions==n_training)&(agg.min_pct>=100*MIN_DIRECT_COVERAGE)]
    choices=[]
    for _,g in agg.groupby(['scale','tau'],sort=False):
        best=g.median_r.max()
        choices.append(g[g.median_r>=best-delta].sort_values('window_days').iloc[0])
    if not choices:
        raise ValueError(f'{holdout}: no transfer candidate survives training coverage rule')
    choices=pd.DataFrame(choices)
    choices['training_score']=lam*(choices.median_r+1)/2+(1-lam)*choices.median_pct/100
    selected=choices.sort_values(['training_score','median_pct','window_days','scale','tau'],
                                 ascending=[False,False,True,True,True]).iloc[0]
    key=(target.scale==selected.scale)&(target.tau==selected.tau)&(target.window_days==selected.window_days)
    if key.sum()!=1:
        raise ValueError(f'{holdout}: selected configuration missing from held-out metric matrix')
    evaluated=target[key].iloc[0]
    baseline=target[(target.scale=='G8')&(target.tau==0)&(target.window_days==1)].iloc[0]
    valid_heldout=bool(np.isfinite(evaluated.r) and evaluated.n_pairs>=30)
    # Hindsight comparison uses held-out load and the SAME near-plateau rule.
    # It is reported only as an optimistic benchmark, never used for selection.
    hindsight=target[target.r.notna()&(target.n_pairs>=30)&
                     (target.n_pct>=100*MIN_DIRECT_COVERAGE)].copy()
    local_choices=[]
    for _,g in hindsight.groupby(['scale','tau'],sort=False):
        best=g.r.max()
        local_choices.append(g[g.r>=best-delta].sort_values('window_days').iloc[0])
    if not local_choices:
        raise ValueError(f'{holdout}: no eligible hindsight comparison')
    local_choices=pd.DataFrame(local_choices)
    local_choices['oracle_score']=lam*(local_choices.r+1)/2+(1-lam)*local_choices.n_pct/100
    oracle=local_choices.sort_values(['oracle_score','n_pct'],ascending=False).iloc[0]

    # Isolate spatial/SC choices: compare selected NTL with G8 at the SAME
    # window, against the SAME smoothed load, and on the SAME calendar dates.
    selected_common_r, g8_common_r, common_n=np.nan,np.nan,0
    if region_data is not None and region_load is not None:
        held_load=region_load[holdout]
        sx,sy,_,_=smooth_pair(region_data[(holdout,selected.scale)],held_load,
                              int(selected.tau),int(selected.window_days),'direct')
        bx,_,_,_=smooth_pair(region_data[(holdout,'G8')],held_load,0,
                             int(selected.window_days),'direct')
        common=held_load.index[sx.notna()&bx.notna()&sy.notna()]
        sm=paired_metrics(sx,sy,common)
        bm=paired_metrics(bx,sy,common)
        selected_common_r,g8_common_r,common_n=sm['r'],bm['r'],sm['n_pairs']
    return {'held_out':holdout,'lambda_alignment':lam,'delta_r':delta,
            'selected_mask':selected.scale,'selected_tau':int(selected.tau),
            'selected_window_days':int(selected.window_days),
            'training_score':float(selected.training_score),
            'held_out_r':evaluated.r,'held_out_rmse_z':evaluated.rmse_z,
            'held_out_n':int(evaluated.n_pairs),'held_out_n_pct':evaluated.n_pct,
            'valid_heldout':valid_heldout,
            'selected_common_r':selected_common_r,'g8_same_window_common_r':g8_common_r,
            'common_pair_n':common_n,
            'baseline_G8_raw_r':baseline.r,'baseline_G8_raw_n_pct':baseline.n_pct,
            'hindsight_oracle_r':oracle.r,'hindsight_oracle_n_pct':oracle.n_pct,
            'hindsight_oracle_mask':oracle.scale,'hindsight_oracle_tau':int(oracle.tau),
            'hindsight_oracle_window_days':int(oracle.window_days)}

TRANSFER=pd.DataFrame()
if not CROSS_MATRIX.empty:
    transfer_rows=[]
    for holdout in CROSS_LOAD:
        for lam in [.3,.5,.7]:
            for delta in [.01,.02,.05]:
                try:
                    transfer_rows.append(transfer_fold(CROSS_MATRIX,holdout,lam,delta,CROSS_DATA,CROSS_LOAD))
                except ValueError as exc:
                    print('Transfer fold skipped:',exc)
    TRANSFER=pd.DataFrame(transfer_rows)
    TRANSFER.to_csv(OUT/'leave_one_island_out_weight_and_window_transfer.csv',index=False)
    print(f'Saved {len(TRANSFER)} held-out evaluations across {len(CROSS_LOAD)} eligible grids.')
    if not TRANSFER.empty and (~TRANSFER.valid_heldout).any():
        print('ATTENTION: Some selected held-out configurations have fewer than 30 pairs or undefined r.')
        display(TRANSFER.loc[~TRANSFER.valid_heldout,
                         ['held_out','lambda_alignment','delta_r','selected_mask','selected_tau',
                          'selected_window_days','held_out_n']])

### Figure 6 — Generalizability is a held-out question

This chart displays the predeclared `λ=0.5, Δr=0.02` policy for each held-out island. It shows held-out $r$ **and** direct-date fraction, alongside the capped G8 τ=0, one-day comparator and the held-out *hindsight* best score (using the same window rule). That whole-pipeline G8 comparison changes smoothing as well as mask. The exported table adds a **same-window, same-date G8 comparison** to isolate the mask/SC contribution. Figure 6B then checks the held-out average across all declared weights and window tolerances; cells are hidden unless all available folds have valid $r$. If the transferred choices remain useful across grids, that supports transport of the **selection logic**; if they vary or underperform, report that limitation and consider a smaller family of application-specific policies. It does **not** justify calculating an $r$-weighted score where load is unavailable.

In [ ]:
if not TRANSFER.empty:
    q=TRANSFER[(TRANSFER.lambda_alignment==.5)&(TRANSFER.delta_r==.02)].copy()
    q=q[q.valid_heldout].copy()
    if q.empty:
        raise ValueError('No valid held-out fold for the declared λ=0.5, Δr=0.02 display')
    order=list(CROSS_LOAD)
    q['held_out']=pd.Categorical(q.held_out,categories=order,ordered=True)
    q=q.sort_values('held_out')
    fig=make_subplots(rows=1,cols=2,subplot_titles=['Held-out Pearson r','Held-out directly paired load dates (%)'],
                      horizontal_spacing=.11)
    for col,triples in [(1,[('G8 τ=0, 1 d','baseline_G8_raw_r','#A8CE74'),
                           ('Transferred rule','held_out_r','#4E79A7'),
                           ('Hindsight local best','hindsight_oracle_r','#D65A4A')]),
                        (2,[('G8 τ=0, 1 d','baseline_G8_raw_n_pct','#A8CE74'),
                            ('Transferred rule','held_out_n_pct','#4E79A7'),
                            ('Hindsight local best','hindsight_oracle_n_pct','#D65A4A')])]:
        for name,key,color in triples:
            fig.add_trace(go.Scatter(x=q.held_out.astype(str),y=q[key],mode='markers+lines',
                                     name=name,showlegend=(col==1),
                                     marker=dict(size=10,color=color),line=dict(color=color,width=1.5)),row=1,col=col)
    fig.update_yaxes(title_text='Pearson r',row=1,col=1)
    fig.update_yaxes(title_text='Directly paired dates (%)',row=1,col=2)
    fig.update_xaxes(title_text='Held-out island grid',tickangle=-20)
    fig.update_layout(template='plotly_white',width=1250,height=510,
                      title='Cross-island transfer of a fixed reliability policy (λ=0.5; Δr=0.02)',
                      legend=dict(orientation='h',y=-.22),margin=dict(l=70,r=30,t=80,b=120),font=dict(color='#233B60'))
    save_fig(fig,'fig_06_held_out_rule_transfer')
    display(q[['held_out','selected_mask','selected_tau','selected_window_days',
               'held_out_r','held_out_n','held_out_n_pct','common_pair_n',
               'selected_common_r','g8_same_window_common_r']].round(3))

    summary=(TRANSFER[TRANSFER.valid_heldout]
             .groupby(['lambda_alignment','delta_r'],as_index=False)
             .agg(mean_heldout_r=('held_out_r','mean'),
                  mean_heldout_n_pct=('held_out_n_pct','mean'),
                  valid_folds=('held_out','nunique')))
    summary.loc[summary.valid_folds<len(CROSS_LOAD),['mean_heldout_r','mean_heldout_n_pct']]=np.nan
    summary.to_csv(OUT/'transfer_priority_sensitivity_summary.csv',index=False)
    fig=make_subplots(rows=1,cols=2,subplot_titles=['Mean held-out r','Mean direct-date retention (%)'],
                      horizontal_spacing=.15)
    for col,field,zmin,zmax,title in [(1,'mean_heldout_r',-1,1,'Pearson r'),
                                     (2,'mean_heldout_n_pct',0,100,'N%')]:
        p=summary.pivot(index='delta_r',columns='lambda_alignment',values=field)
        fig.add_trace(go.Heatmap(x=p.columns,y=p.index,z=p.values,colorscale='Blues',
                                 zmin=zmin,zmax=zmax,colorbar=dict(title=title),
                                 hovertemplate='λ=%{x}<br>Δr=%{y}<br>Mean=%{z:.3f}<extra></extra>'),row=1,col=col)
    fig.update_xaxes(title_text='Alignment priority λ')
    fig.update_yaxes(title_text='Near-plateau tolerance Δr',row=1,col=1)
    fig.update_layout(template='plotly_white',width=1150,height=470,
                      title=f'Held-out sensitivity across priorities (complete {len(CROSS_LOAD)}-fold cells only)',
                      margin=dict(l=80,r=80,t=80,b=70),font=dict(color='#233B60'))
    save_fig(fig,'fig_06b_held_out_priority_sensitivity')
else:
    print('Figure 6 awaits the RQ1 Notebook 04 cross-island input folder and NGCP workbook; no transfer claim is made.')

## Where the figures belong

**Main paper:** Figure 1A–B (2012–2024 baseline from the paired Region VIII CSV); Figure 2 (why SC and GHSL are distinct); Figure 3 (1–30-day $r$ and complete-window N% at τ=0/10/50); Figure 5B–D (two-axis priority and 231-combination composite-score robustness). **Supplement:** Haiyan and Rai zooms; RMSE curves; all-mask heatmaps; complete sensitivity matrices; held-out transfer Figure 6 unless the transfer result becomes a central claim.

The 2012–2024 window and the geographic footprint must be stated in every caption. Figure 4's direct-N% heatmap varies with mask and SC threshold; complete-window N% decreases with smoothing width but is **not** direct daily observation. The manuscript's original 30 retained observations and this notebook's 30 calendar days are distinct. No results have been pre-interpreted here.